# Elemental composition

Here we have a code to compute the elemental formula from a given mass 
<br>
note that this rely on <br>
    1. assuming the mass is accurate 
    2. your given element restriction is correct (i.e., no S or halogens)<br>
    3. this also assumes that the m/z value is from adding or removing proton (i.e., M+H or M-H), you could throw in neutral mass and make the "mode" parameter to be "neutral"<br>
    4. use nitrogen rule and DBE filter on your own risk, as some higher mass species can be problematic<br>
    5. this will try to match Na if M+H fails. this is not the right approach but better than nothing lol<br>

In [ ]:
from molmass import Formula
from pyopenms import ElementDB

In [ ]:
def compute_formula_from_mass_efficient(target, tol=5.0, tol_unit="ppm", mode="positive", elements=None, apply_nitrogen_rule=False, apply_hc_ratio=False, return_tuple=True, force_integer_DBE=False):
    """
    Computes molecular formula from a given mass with high efficiency using dynamic nested loops (recursion) 
    and fast calculation for the lightest element.
    
    Args:
        target (float): The input m/z value.
        tol (float): Tolerance value.
        tol_unit (str): Unit of tolerance ("ppm" or "Da").
        mode (str): Ionization mode ("positive", "pos", "negative", "neg").
        elements (dict): Dictionary of element limits. Default: {'C': 50, 'H': 100, 'N': 10, 'O': 20, 'S': 2}.
                         This defines BOTH the allowed elements and their maximum counts.
        apply_nitrogen_rule (bool): Whether to apply the Nitrogen Rule (valid for mass < 500).
        apply_hc_ratio (bool): Whether to enforce H/C <= 3.

    Returns:
        tuple or str or None: The (formula string, mass) of the best match, or "NA" if no match found.
    """
    
    # 1. Setup Elements and Masses n 
    edb = ElementDB()
    default_limits = {'C': 50, 'H': 100, 'N': 10, 'O': 20, 'S': 1}
    
    if elements is None:
        limits = default_limits
    else:
        limits = elements
        
    # Validation and Pre-processing
    element_data = [] # List of dicts: {'el': symbol, 'mass': mass, 'limit': limit}
    
    try:
        for el, limit in limits.items():
            try:
                msg = edb.getElement(el)
                mono_mass = msg.getMonoWeight()
                nom_mass =  Formula(el).nominal_mass
            except:
                raise ValueError(f"Invalid element symbol: {el}")
            
            element_data.append({
                'el': el,
                'mass': mono_mass,
                'limit': limit,
                'nom_mass': nom_mass
            })
    except ValueError as e:
        print(f"Error initializing elements: {e}")
        return "Error for elements"

    if not element_data:
        print("No elements provided.")
        return "NA"

    # Strategy: 
    # 1. Identify "Calculation Element" -> The lightest one (usually H).
    #    We calculate this directly instead of iterating to save 100x iterations.
    # 2. Identify "Iteration Elements" -> The rest.
    #    Sort them by mass Descending (Heaviest first).
    #    This allows early pruning in the recursion (breaking early if mass exceeded).
    
    # Find lightest
    element_data.sort(key=lambda x: x['mass'])
    calc_element = element_data[0] # Lightest
    iteration_elements = element_data[1:]
    
    # Sort iteration elements by mass Descending
    iteration_elements.sort(key=lambda x: x['mass'], reverse=True)
    
    # Pre-calculate constants
    mass_proton = 1.0072764666
    mass_electron = 0.00054858
    try:
        mass_Na = edb.getElement("Na").getMonoWeight()
    except:
        mass_Na = 22.989769 # Fallback if Na not in user list logic? No, independent lookup.
        
    calc_el_mass = calc_element['mass']
    calc_el_limit = calc_element['limit']
    calc_el_symbol = calc_element['el']

    # 2. Define Helper for Search (Recursive)
    def search_formulas(target_neutral_mass):
        # Calculate tolerance in Da
        if tol_unit == "ppm":
            tolerance_da = target_neutral_mass * tol / 1e6
        else:
            tolerance_da = tol
        
        candidates = []
        
        # Recursive function
        # level: index in iteration_elements
        # current_mass: methods accumulated so far
        # current_counts: list of counts matching iteration_elements order
        
        def recurse(level, current_mass, current_counts):
            # Check if we exceeded target + tolerance (Pruning)
            # Even 0 of remaining elements adds 0 mass. 
            # If current > target + tol, stop.
            if current_mass > target_neutral_mass + tolerance_da:
                return

            # Base Case: All iteration elements handled
            if level == len(iteration_elements):
                # Now determine required amount of calc_element
                remaining_mass = target_neutral_mass - current_mass
                
                # We need: remaining_mass ~= count * calc_el_mass
                count_est = remaining_mass / calc_el_mass
                count = int(round(count_est))
                
                if count < 0:
                    return # Overshot
                if count > calc_el_limit:
                    return # Too many needed
                
                # Check final mass
                final_mass = current_mass + count * calc_el_mass
                diff = abs(final_mass - target_neutral_mass)
                
                if diff <= tolerance_da:
                    # Valid match found!
                    # Construct full counts dictionary
                    full_counts = {calc_el_symbol: count}
                    for i, el_info in enumerate(iteration_elements):
                        full_counts[el_info['el']] = current_counts[i]
                    
                    # --- Filtering ---
                    
                    # 1. H/C Ratio
                    if apply_hc_ratio:
                        # Need C and H in the formula to check.
                        # If user didn't provide H or C, we skip or assume valid?
                        # Usually logic applies if both exist.
                        c_count = full_counts.get('C', 0)
                        h_count = full_counts.get('H', 0)
                        
                        if c_count > 0:
                            if h_count / c_count > 4:
                                return
                        if c_count == 0 and h_count > 0:
                            # If no C but has H, this is likely invalid for typical organic molecules.
                            return
                    
                    # 2. Nitrogen Rule
                    # Mass < 500, N parity == H parity (roughly, for odd electron/valency logic)
                    # if apply_nitrogen_rule and target_neutral_mass < 500:
                    #     n_count = full_counts.get('N', 0)
                    #     h_count = full_counts.get('H', 0)
                    #     # Valid only if we know about N and H
                    #     if (n_count % 2) != (h_count % 2):
                    #         return
                    if apply_nitrogen_rule:
                        n_count = full_counts.get('N', 0)
                        
                        # Calculate the mass of the calc_element (Hydrogen)
                        calc_el_nom_mass = full_counts.get(calc_el_symbol, 0) * calc_element['nom_mass']
                        
                        # Calculate the mass of the iteration elements (C, N, O, S)
                        iter_el_nom_mass = sum(full_counts[el_info['el']] * el_info['nom_mass'] for el_info in iteration_elements)
                        
                        # Total neutral nominal mass
                        nom_mass = calc_el_nom_mass + iter_el_nom_mass
                        
                        # Now apply the classic Nitrogen Rule safely
                        if not (nom_mass % 2 == n_count % 2):
                            return
                        
                    # 3. DBE (Double Bond Equivalent)
                    # note that DBE for neutral compounds is usually integer but set to optional only
                    # also we dont consider any halogens at the momemnt 
                    c_count = full_counts.get('C', 0)
                    h_count = full_counts.get('H', 0)
                    n_count = full_counts.get('N', 0)
                    # Calculate DBE
                    dbe = c_count + 1 - (h_count / 2.0) + (n_count / 2.0)
                    if dbe < 0:
                        return 

                    if force_integer_DBE and dbe != int(dbe):
                        return

                    # Add to candidates
                    # Format for formula string: we need a standardized order?
                    # Usually Hill system: C then H then alphabetical.
                    # Helper `get_formula_string` expects a list `counts` and list `elements`
                    # We should reconstruct generic lists.
                    
                    # Let's create sorted list of all elements for string generation
                    all_elements_keys = sorted(full_counts.keys())
                    # Hill system partial sort: Put C first, then H, then rest alphabetical
                    sorted_keys = []
                    if 'C' in all_elements_keys:
                        sorted_keys.append('C')
                        all_elements_keys.remove('C')
                    if 'H' in all_elements_keys:
                        sorted_keys.append('H')
                        all_elements_keys.remove('H')
                    sorted_keys.extend(all_elements_keys)
                    
                    final_counts_list = [full_counts[k] for k in sorted_keys]
                    
                    candidates.append({
                        'formula_str': get_formula_string(final_counts_list, elements=sorted_keys),
                        'mass': final_mass,
                        'error': diff
                    })
                return

            # Recursive Step
            # Iterate current element from 0 to limit
            el_info = iteration_elements[level]
            el_mass = el_info['mass']
            el_limit = el_info['limit']
            
            for c in range(el_limit + 1):
                new_mass = current_mass + c * el_mass
                # Pruning check before recursion
                if new_mass > target_neutral_mass + tolerance_da:
                    break # Since we are adding positive mass, further counts will also fail
                
                # Recurse
                # We extend the counts list. 
                # Optimization: pass list copy or append/pop?
                # Append/pop is faster in Python generally if depth is high, but pass-by-value (list + [c]) is safer/cleaner code.
                recurse(level + 1, new_mass, current_counts + [c])

        # Start Recursion
        recurse(0, 0.0, [])
        return candidates

    # 3. Main Logic with Fallbacks
    
    primary_neutral_mass = None
    if mode == "positive" or mode =="pos":
        primary_neutral_mass = target - mass_proton
    elif mode == "negative" or mode =="neg":
        primary_neutral_mass = target + mass_proton
    else:
        primary_neutral_mass = target
        
    # Attempt 1: Primary Search
    results = search_formulas(primary_neutral_mass)
    
    if results:
        results.sort(key=lambda x: x['error'])
        best = results[0]
        if not return_tuple:
            return best['formula_str'], best['mass']
        else:            
           return (best['formula_str'], best['mass'])
        
    # Fallback Handling
    fallback_neutral_mass = None
    
    if mode == "positive" or mode =="pos":
        # Check Sodium Adduct [M+Na]+ -> Neutral M = Target - Na_ion
        mz_sodium_ion = mass_Na - mass_electron
        fallback_neutral_mass = target - mz_sodium_ion
        
        results_na = search_formulas(fallback_neutral_mass)
        if results_na:
            results_na.sort(key=lambda x: x['error'])
            best = results_na[0]
            if not return_tuple:
                return best['formula_str'], best['mass']
            else:            
                return (best['formula_str'], best['mass']) # Return neutral formula
            
    elif mode == "negative" or mode =="neg":
        # Check Dimer [2M-H]- -> Neutral M = (Target + H_ion) / 2
        fallback_neutral_mass = (target + mass_proton) / 2.0
        
        results_dimer = search_formulas(fallback_neutral_mass)
        if results_dimer:
            results_dimer.sort(key=lambda x: x['error'])
            best = results_dimer[0]
            if not return_tuple:
                return best['formula_str'], best['mass']
            else:            
                return (best['formula_str'], best['mass'])
    if return_tuple:
        return (None, None)
    else:         
        return None, None